# Healome Aging Clock — Training Notebook

This notebook reproduces the full training pipeline for the Healome Aging Clock,
a blood-based biological age model using Gradient Boosted Trees trained on NHANES data.

**Sections:**
1. Setup and data loading
2. Feature engineering and preprocessing
3. Model training — 21-feature standard model
4. Model training — 35-feature extended model
5. Feature importance analysis
6. Mortality data and survival analysis
7. Kaplan-Meier survival curves
8. Cox Proportional Hazards model
9. Save trained models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from os import listdir
from pathlib import Path

from sklearn import ensemble
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from joblib import dump, load

import sys
sys.path.insert(0, '..')

from healome_clock.models.tree import (
    STANDARD_21_FEATURES, EXTENDED_35_FEATURES, FEATURE_NAMES
)
from healome_clock.evaluation.metrics import compute_age_metrics, print_metrics

import warnings
warnings.filterwarnings('ignore')

## 1. Load NHANES Data

We load and merge SAS Transport (XPT) files from multiple NHANES survey cycles.
The primary cycle is 2017-2020 (pre-pandemic), with extended data from 2003-2016.

In [ ]:
# --- Primary dataset: NHANES 2017-2020 (pre-pandemic) ---
# Update this path to your local NHANES data directory
path_prefix = "../path/to/NHANES/2017-2020"

# Load and merge key data files
dd_age = pd.read_sas(os.path.join(path_prefix, "P_BIOPRO.XPT"), format="XPORT")
dd_age = dd_age.merge(pd.read_sas(os.path.join(path_prefix, "P_MCQ.XPT"), format="XPORT"), how="outer", on=["SEQN"])
dd_age = dd_age.merge(pd.read_sas(os.path.join(path_prefix, "P_TRIGLY.XPT"), format="XPORT"), how="outer", on=["SEQN"])
dd_age = dd_age.merge(pd.read_sas(os.path.join(path_prefix, "P_HSCRP.XPT"), format="XPORT"), on=["SEQN"], how="outer")
dd_age = dd_age.merge(pd.read_sas(os.path.join(path_prefix, "P_HDL.XPT"), format="XPORT"), on=["SEQN"], how="outer")
dd_age = dd_age.merge(pd.read_sas(os.path.join(path_prefix, "P_CBC.XPT"), format="XPORT"), on=["SEQN"], how="outer")
dd_age = dd_age.merge(pd.read_sas(os.path.join(path_prefix, "P_GHB.XPT"), format="XPORT"), on=["SEQN"], how="outer")
dd_age = dd_age.merge(pd.read_sas(os.path.join(path_prefix, "P_DEMO.XPT"), format="XPORT"), on=["SEQN"], how="outer")

print(f"Primary dataset: {dd_age.shape[0]} records, {dd_age.shape[1]} columns")

In [ ]:
# --- Extended data: earlier NHANES cycles (2003-2016) ---
# Update these paths to your local data
path_prefixes = [
    "../path/to/NHANES/extended_data/2015-16",
    "../path/to/NHANES/extended_data/2013-14",
    "../path/to/NHANES/extended_data/2011-12",
    "../path/to/NHANES/extended_data/2009-10",
    "../path/to/NHANES/extended_data/2007-08",
    "../path/to/NHANES/extended_data/2005-06",
    "../path/to/NHANES/extended_data/2003-04",
]

for pfx in path_prefixes:
    if not os.path.exists(pfx):
        print(f"  Skipping {pfx} (not found)")
        continue
    new_dd_age = None
    for file in listdir(pfx):
        try:
            df = pd.read_sas(os.path.join(pfx, file), format="xport")
            if new_dd_age is None:
                new_dd_age = df
            else:
                new_dd_age = new_dd_age.merge(df, how="outer")
        except Exception:
            pass
    if new_dd_age is not None:
        dd_age = pd.concat([dd_age, new_dd_age], ignore_index=True)
        print(f"  Added {pfx}: total now {dd_age.shape[0]} records")

print(f"\nFinal dataset: {dd_age.shape[0]} records, {dd_age.shape[1]} columns")

## 2. Preprocessing

Apply mean imputation and prepare feature matrices.

In [ ]:
# Impute missing values with column means
dd_age = dd_age.fillna(dd_age.mean())

print(f"Age range: {dd_age['RIDAGEYR'].min():.0f} - {dd_age['RIDAGEYR'].max():.0f} years")
print(f"Mean age: {dd_age['RIDAGEYR'].mean():.1f} years")
print(f"Records: {len(dd_age):,}")

## 3. Train: 21-Feature Standard Model

16 lab biomarkers + 5 questionnaire items.
GradientBoostingRegressor with n_estimators=4000, max_depth=8.

In [ ]:
imp_adhoc = STANDARD_21_FEATURES
print(f"Features ({len(imp_adhoc)}):")
for f in imp_adhoc:
    print(f"  {f:12s}  {FEATURE_NAMES.get(f, '')}")

x = np.array(dd_age.loc[:, imp_adhoc])
y = np.array(dd_age.loc[:, "RIDAGEYR"])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=3454
)
print(f"\nTrain: {x_train.shape[0]:,}  Test: {x_test.shape[0]:,}")

In [ ]:
params_standard = {
    "n_estimators": 4000,
    "max_depth": 8,
    "min_samples_split": 30,
    "learning_rate": 0.01,
    "loss": "ls",
}

a_clock = ensemble.GradientBoostingRegressor(**params_standard)
a_clock.fit(x_train, y_train)

print("Training accuracy")
train_metrics = compute_age_metrics(y_train, a_clock.predict(x_train))
print_metrics(train_metrics, "21-Feature Standard — Train")

print("Testing accuracy")
test_metrics = compute_age_metrics(y_test, a_clock.predict(x_test))
print_metrics(test_metrics, "21-Feature Standard — Test")

### Feature Importance — 21-Feature Model

In [ ]:
feature_importance = a_clock.feature_importances_
sorted_idx = np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + 0.5

feature_labels = np.array([
    f"{FEATURE_NAMES.get(f, f)} ({f})" for f in imp_adhoc
])

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(pos, feature_importance[sorted_idx], align="center", color="#2563EB", alpha=0.85)
ax.set_yticks(pos)
ax.set_yticklabels(feature_labels[sorted_idx], fontsize=10)
ax.set_xlabel("Feature Importance", fontsize=12)
ax.set_title("Feature Importance — 21-Feature Standard Model", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/feature_importance_21feat.png", dpi=150, bbox_inches="tight")
plt.show()

print("Feature importance order (least → most):")
print(np.array(imp_adhoc)[sorted_idx])

## 4. Train: 35-Feature Extended Model

Adds lipids, liver markers, hematologic markers.
GradientBoostingRegressor with n_estimators=6000, max_depth=10.

In [ ]:
imp_adhoc_ext = EXTENDED_35_FEATURES
print(f"Features ({len(imp_adhoc_ext)}):")

x = np.array(dd_age.loc[:, imp_adhoc_ext])
y = np.array(dd_age.loc[:, "RIDAGEYR"])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=3454
)
print(f"Train: {x_train.shape[0]:,}  Test: {x_test.shape[0]:,}")

In [ ]:
params_extended = {
    "n_estimators": 6000,
    "max_depth": 10,
    "min_samples_split": 30,
    "learning_rate": 0.01,
    "loss": "ls",
}

a_clock_ext = ensemble.GradientBoostingRegressor(**params_extended)
a_clock_ext.fit(x_train, y_train)

print("Training accuracy")
train_metrics = compute_age_metrics(y_train, a_clock_ext.predict(x_train))
print_metrics(train_metrics, "35-Feature Extended — Train")

print("Testing accuracy")
test_metrics = compute_age_metrics(y_test, a_clock_ext.predict(x_test))
print_metrics(test_metrics, "35-Feature Extended — Test")

### Feature Importance — 35-Feature Model

In [ ]:
feature_importance = a_clock_ext.feature_importances_
sorted_idx = np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + 0.5

feature_labels = np.array([
    f"{FEATURE_NAMES.get(f, f)} ({f})" for f in imp_adhoc_ext
])

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(pos, feature_importance[sorted_idx], align="center", color="#DC2626", alpha=0.85)
ax.set_yticks(pos)
ax.set_yticklabels(feature_labels[sorted_idx], fontsize=9)
ax.set_xlabel("Feature Importance", fontsize=12)
ax.set_title("Feature Importance — 35-Feature Extended Model", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/feature_importance_35feat.png", dpi=150, bbox_inches="tight")
plt.show()

print("Feature importance order (least → most):")
print(np.array(imp_adhoc_ext)[sorted_idx])

## 5. Predicted vs. Actual Age Plots

In [ ]:
from healome_clock.visualization import plot_predicted_vs_actual

# 21-feature model
x_21 = np.array(dd_age.loc[:, STANDARD_21_FEATURES])
y_all = np.array(dd_age.loc[:, "RIDAGEYR"])
_, x_test_21, _, y_test_21 = train_test_split(x_21, y_all, test_size=0.3, random_state=3454)

fig = plot_predicted_vs_actual(
    y_test_21, a_clock.predict(x_test_21),
    title="Predicted vs. Actual Age — 21-Feature Model",
    save_path="../figures/predicted_vs_actual_21feat.png"
)
plt.show()

In [ ]:
# 35-feature model
x_35 = np.array(dd_age.loc[:, EXTENDED_35_FEATURES])
_, x_test_35, _, y_test_35 = train_test_split(x_35, y_all, test_size=0.3, random_state=3454)

fig = plot_predicted_vs_actual(
    y_test_35, a_clock_ext.predict(x_test_35),
    title="Predicted vs. Actual Age — 35-Feature Model",
    save_path="../figures/predicted_vs_actual_35feat.png"
)
plt.show()

## 6. Load Mortality Data

Download NHANES linked mortality files from the CDC FTP server.
These provide vital status (alive/dead) and time-to-event for each participant.

In [ ]:
from healome_clock.data.mortality import load_mortality_data, merge_with_mortality

# Download and parse mortality data for all available cycles
mortality = load_mortality_data(
    years=[2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018],
    download=True,
)
mortality.head()

In [ ]:
# Add biological age predictions to the data
dd_age["bio_age"] = a_clock.predict(np.array(dd_age.loc[:, STANDARD_21_FEATURES]))

# Merge with mortality data
anal_data = merge_with_mortality(dd_age, mortality)
print(f"\nRecords with mortality data: {len(anal_data):,}")
print(f"Deceased: {anal_data['is_dead'].sum():,}")

## 7. Kaplan-Meier Survival Analysis

Compare survival curves for individuals classified as:
- **Accelerated aging**: biological age >= chronological age + 5 years
- **Decelerated aging**: biological age <= chronological age - 5 years

In [ ]:
from healome_clock.evaluation.survival import (
    prepare_survival_data, compute_kaplan_meier
)

surv_data = prepare_survival_data(anal_data, threshold=5.0)

print(f"Accelerated aging: {(surv_data['aging_rate'] == 'accelerated').sum():,}")
print(f"Normal aging:      {(surv_data['aging_rate'] == 'normal').sum():,}")
print(f"Decelerated aging: {(surv_data['aging_rate'] == 'decelerated').sum():,}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
km_results = compute_kaplan_meier(
    surv_data,
    duration_col="chrono_age_at_event",
    event_col="is_dead",
    groups=["accelerated", "decelerated"],
    ax=ax,
)
plt.savefig("../figures/kaplan_meier_aging_rate.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Cox Proportional Hazards

Fit a Cox PH model to quantify the hazard ratio of biological vs chronological age.

In [ ]:
from healome_clock.evaluation.survival import compute_cox_hazard_ratios

cox_results = compute_cox_hazard_ratios(
    surv_data,
    duration_col="years_until_death",
    event_col="is_dead",
    covariates=["RIDAGEYR", "bio_age"],
)

print(f"Concordance Index: {cox_results['concordance']:.4f}")
print(f"\nHazard Ratios:")
print(cox_results['hazard_ratios'])
print(f"\nFull Summary:")
cox_results['model'].print_summary()

## 9. Save Trained Models

In [ ]:
# Save the 21-feature standard model
dump(a_clock, '../healome_clock/models/weights/standard_21feat.joblib')
print("Saved: standard_21feat.joblib")
print(f"Feature order: {STANDARD_21_FEATURES}")

# Save the 35-feature extended model
dump(a_clock_ext, '../healome_clock/models/weights/extended_35feat.joblib')
print("\nSaved: extended_35feat.joblib")
print(f"Feature order: {EXTENDED_35_FEATURES}")